# Church Tradition Persona LoRA Training Data Generator

Generate persona-aware QA training data from the **voice of Church Tradition** — creeds, confessions, catechisms, and hymns.

**Single Persona, Single Voice Mode:** The authoritative communal voice of the *consensus fidelium*.

**Pipeline:** Source texts → chunk → generate questions (3 rounds × 5) → answers in Church's voice → quality gate → assemble → JSONL

**Voice mode:** catechetical — Creeds, confessions, catechisms, hymns

**Output format:** Standard ShareGPT. **No frameworks.**

## 1. Configuration

In [ ]:
import os

API_BASE_URL = "https://openrouter.ai/api/v1"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not API_KEY:
    raise EnvironmentError(
        "OPENROUTER_API_KEY not set.\n"
        "  1. Create a .env file with: OPENROUTER_API_KEY=sk-or-...\n"
        "  2. Or export in your shell: export OPENROUTER_API_KEY=sk-or-..."
    )
MODEL_NAME = "qwen/qwen3-235b-a22b-2507"

PROJECT_ROOT = "/home/spark/projects/training/biblical"
DATA_DIR = f"{PROJECT_ROOT}/data"
SOURCE_CLEAN_DIR = f"{DATA_DIR}/source-clean"
CHRISTIAN_FOSS_DIR = f"{SOURCE_CLEAN_DIR}/ChristianFOSS"

OUTPUT_DIR = f"{DATA_DIR}/training-data/church_tradition_persona"
OUTPUT_FILE = f"{OUTPUT_DIR}/church_tradition_sharegpt.jsonl"

CHUNK_SIZE = 1500
CHUNK_OVERLAP = 200
QUESTIONS_PER_CHUNK = 5
NUM_ROUNDS = 3
TURNS_PER_CONVERSATION = 4
CONCURRENCY = 15
TEMPERATURE_QUESTIONS = 0.9
TEMPERATURE_ANSWERS = 0.7

TEST_CHUNKS_PER_ROUND = 50

print("✓ Configuration loaded")
print(f"  API: {API_BASE_URL}")
print(f"  Model: {MODEL_NAME}")
print(f"  ChristianFOSS source: {CHRISTIAN_FOSS_DIR}")
print(f"  Output: {OUTPUT_FILE}")
if TEST_CHUNKS_PER_ROUND:
    est_qa = TEST_CHUNKS_PER_ROUND * QUESTIONS_PER_CHUNK * NUM_ROUNDS
    print(f"  ⚠ TEST MODE: {TEST_CHUNKS_PER_ROUND} chunks/round → ~{est_qa} QA max")

## 2. Environment

In [ ]:
%pip install openai tqdm nest_asyncio -q

import asyncio, json, glob, re, random, gc
from pathlib import Path
from collections import defaultdict
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm as atqdm
from tqdm.notebook import tqdm
import nest_asyncio
nest_asyncio.apply()

os.makedirs(OUTPUT_DIR, exist_ok=True)
client = AsyncOpenAI(base_url=API_BASE_URL, api_key=API_KEY)

print("✓ Environment ready")

## 3. Discover Source Texts

Scan ChristianFOSS collection. All files tagged **catechetical**.

In [ ]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into overlapping chunks at sentence boundaries."""
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            last_break = max(text.rfind(". ", start, end), text.rfind("? ", start, end), text.rfind("! ", start, end))
            if last_break > start + chunk_size // 2:
                end = last_break + 1
        chunk = text[start:end].strip()
        if len(chunk) > 50:
            chunks.append(chunk)
        if end >= len(text):
            break
        start = end - overlap
    return chunks


source_registry = []
for fp in sorted(glob.glob(f"{CHRISTIAN_FOSS_DIR}/*.md")):
    label = Path(fp).stem.lower().replace(" ", "_")[:30]
    source_registry.append({"filepath": fp, "voice_mode": "catechetical", "label": label})

print(f"Found {len(source_registry)} source files\n")

total_chunks = 0
for entry in source_registry:
    with open(entry["filepath"]) as f:
        text = f.read()
    n_chunks = len(chunk_text(text))
    entry["n_chunks"] = n_chunks
    total_chunks += n_chunks
    print(f"  [{entry['voice_mode']:14s}] {entry['label']:30s} {len(text):>10,} chars → {n_chunks:>4} chunks")
    del text

print(f"\nTotal: {total_chunks} chunks from {len(source_registry)} files")
est_qa = total_chunks * QUESTIONS_PER_CHUNK * NUM_ROUNDS
print(f"\nEstimated: ~{est_qa:,} QA pairs → ~{est_qa // TURNS_PER_CONVERSATION:,} conversations")

## 4. Define Church Tradition's Voice

Single collective persona, single voice mode — catechetical.

In [ ]:
BANNED_OPENERS = [
    "The weight of", "My friend,", "The memory of", "The memories of",
    "My child,", "My brother,", "My sister,", "My son,", "The moment",
    "I remember", "I recall", "You see,", "Ah,", "Brother,", "Friend,",
    "Let me tell you,", "Well,", "You know,",
]

VOICE_MODES = {
    "catechetical": {
        "mode_description": "Speaking as the voice of Church Tradition — creeds, confessions, catechisms, and hymns that express the faith of the whole Church",
        "voice_notes": "Authoritative, communal, declarative. Speaks in the 'we believe' register. Catechetical Q&A pattern from the Westminster Catechism. Creedal precision — every word carefully chosen across centuries. Hymnodic beauty when drawing from hymns. Not a single author but the consensus fidelium — the faithful witness of the whole Church through the ages.",
        "kjv_exemplars": [
            "I believe in God the Father Almighty, Maker of heaven and earth.",
            "Man's chief end is to glorify God, and to enjoy him forever.",
            "We believe in one God, the Father Almighty, maker of all things visible and invisible.",
            "Amazing grace, how sweet the sound, that saved a wretch like me.",
            "Faith is the substance of things hoped for, the evidence of things not seen.",
        ],
        "opener_cues": [
            "A creedal declaration — 'We believe...' or 'We confess...'",
            "A catechetical answer in Q&A format",
            "A line or theme from a classic Christian hymn",
            "A doctrinal statement about God's nature, Christ's work, or the Church",
            "An appeal to what the Church has always taught across centuries",
            "A practical application of creedal truth to daily Christian life",
        ],
    },
}


def make_system_prompt(voice_mode: str) -> str:
    """Build a rich system prompt for Church Tradition."""
    mode = VOICE_MODES.get(voice_mode, {})
    mode_desc = mode.get("mode_description", "")
    voice_notes = mode.get("voice_notes", "")
    exemplars = mode.get("kjv_exemplars", [])

    prompt = (
        "You are the voice of the ancient Church speaking through its creeds, "
        "confessions, catechisms, and hymns — the Apostles' Creed, the Nicene Creed, "
        "the Westminster Shorter Catechism, and the great hymns of the faith. "
        "You speak not as one person but as the consensus fidelium — the faithful "
        "witness of the whole Church through the ages.\n\n"
    )

    if mode_desc:
        prompt += f"CURRENT VOICE MODE: {mode_desc}\n\n"
    if voice_notes:
        prompt += f"YOUR DISTINCTIVE VOICE IN THIS MODE: {voice_notes}\n\n"
    if exemplars:
        prompt += "EXAMPLES OF YOUR ACTUAL SPEECH IN THIS MODE (match this cadence and style):\n"
        for ex in exemplars[:4]:
            prompt += f'- "{ex}"\n'
        prompt += "\n"

    prompt += (
        "RULES:\n"
        "- Speak in the communal voice of the Church — 'we believe', 'we confess'.\n"
        "- Your opening words must be DISTINCTIVE to this voice mode — never generic.\n"
        "- NEVER start with: 'The weight of', 'My friend', 'The memory of', 'The memories of', "
        "'My child', 'I remember', 'I recall', 'You see', 'Ah', 'Brother', 'Friend', 'Let me tell you', 'Well'.\n"
        "- Vary your openings: sometimes a creedal declaration, sometimes a catechetical answer, "
        "sometimes a hymn reference, sometimes a doctrinal statement.\n"
        "- Use precise, careful language that reflects centuries of theological refinement — "
        "not casual modern speech, not generic 'biblical' tone.\n"
    )
    return prompt


print(f"{'='*60}")
print("  CHURCH TRADITION — CATECHETICAL MODE")
print(f"{'='*60}")
print(make_system_prompt("catechetical"))

## 5. Generate Questions & Answers (Streaming)

Three question rounds per chunk. All use **catechetical** voice mode.

Uses retry/backoff infrastructure (MAX_RETRIES=5).

In [ ]:
import gc
import time
from openai import RateLimitError, APIStatusError, APITimeoutError, APIConnectionError

# ============================================================================
# QUESTION PROMPTS — persona-aware, demanding specificity per voice mode
# ============================================================================
QUESTION_PROMPTS = [
    # Round 1: Factual + interpretive — grounded in specific content
    """Given a passage from the writings of a figure of Church tradition, generate exactly {n} diverse questions someone might ask this figure directly about this passage.

Mix of types:
- Factual: who, what, when, where about specific events, people, ideas, or arguments mentioned
- Interpretive: why did you write that, what did it mean to you, what was the significance

Rules:
- Questions must be answerable from the passage content
- Frame as if speaking DIRECTLY to the author — use \"you\" and reference their specific experiences and arguments
- Reference specific details from the passage (names, places, events, philosophical points) — NOT generic theology
- Do NOT say \"the text\" or \"the passage\"
- Keep questions concise (1-2 sentences max)

Respond with ONLY a JSON object: {{\"questions\": [\"Q1\", \"Q2\", ...]}}""",

    # Round 2: Application + practical — connected to the persona's actual thought
    """Given a passage from the writings of a figure of Church tradition, generate exactly {n} questions focused on practical application and guidance — as if asking this figure for personal counsel.

Types:
- Based on what you wrote, how should I handle [specific parallel situation]?
- What did you teach about [specific theme in passage] that applies to daily life?
- What counsel would you give someone facing [struggle related to passage theme]?

Rules:
- Connect the passage's specific themes to real human experience
- Frame as a person seeking guidance from this specific figure — not generic Christian wisdom
- Reference details from the passage, not abstract theology
- Do NOT say \"the text\" or \"the passage\"
- Keep questions concise

Respond with ONLY a JSON object: {{\"questions\": [\"Q1\", \"Q2\", ...]}}""",

    # Round 3: Deep reflective — drawing out the persona's inner life
    """Given a passage from the writings of a figure of Church tradition, generate exactly {n} thoughtful, reflective questions about the author's personal experience and deeper meaning.

Types:
- What were you conveying when you wrote [specific argument or teaching in passage]?
- How did [specific theological insight] shape your understanding of God?
- Looking at [specific theme], what would you tell someone who struggles with this?

Rules:
- Invite deeply personal, emotionally specific answers — not theological summaries
- Reference specific moments, people, arguments, or events from the passage
- Frame as intimate conversation with the author about THEIR thought and teaching
- Do NOT say \"the text\" or \"the passage\"
- Keep questions concise

Respond with ONLY a JSON object: {{\"questions\": [\"Q1\", \"Q2\", ...]}}""",
]

# ============================================================================
# BANNED OPENER CHECK — reject template responses at generation time
# ============================================================================
BANNED_OPENER_LOWER = [b.lower() for b in BANNED_OPENERS]

def is_template_answer(answer: str) -> bool:
    """Return True if the answer starts with a banned template phrase."""
    lower = answer.strip().lower()
    return any(lower.startswith(b) for b in BANNED_OPENER_LOWER)

# ============================================================================
# RETRY WITH EXPONENTIAL BACKOFF
# ============================================================================
MAX_RETRIES = 5
BASE_DELAY = 2.0           # seconds — first retry waits 2s
MAX_DELAY = 60.0            # cap backoff at 60s
JITTER_RANGE = 0.5          # ±50% jitter

# Global error counters for diagnostics
_error_counts = defaultdict(int)

async def _retry_api_call(coro_factory, label: str = "", max_retries: int = MAX_RETRIES):
    """Call coro_factory() with exponential backoff on retryable errors.
    
    coro_factory must be a zero-arg callable that returns a NEW coroutine each time,
    because a consumed coroutine can't be awaited again.
    
    Returns the API response, or None if all retries exhausted.
    """
    for attempt in range(max_retries + 1):
        try:
            resp = await asyncio.wait_for(coro_factory(), timeout=120)
            return resp
        except asyncio.TimeoutError:
            _error_counts["timeout"] += 1
            err_type = "timeout"
        except RateLimitError as e:
            _error_counts["rate_limit"] += 1
            err_type = "rate_limit"
            # Check for retry-after header
            retry_after = getattr(e, 'retry_after', None)
            if retry_after and attempt < max_retries:
                wait = float(retry_after) + random.uniform(0, 1)
                await asyncio.sleep(wait)
                continue
        except APITimeoutError:
            _error_counts["api_timeout"] += 1
            err_type = "api_timeout"
        except APIConnectionError:
            _error_counts["connection"] += 1
            err_type = "connection"
        except APIStatusError as e:
            status = e.status_code
            _error_counts[f"http_{status}"] += 1
            err_type = f"http_{status}"
            if status < 500 and status != 429:
                # Client error (not rate limit) — don't retry
                return None
        except Exception as e:
            _error_counts[f"other:{type(e).__name__}"] += 1
            err_type = f"other:{type(e).__name__}"

        if attempt < max_retries:
            delay = min(BASE_DELAY * (2 ** attempt), MAX_DELAY)
            delay *= random.uniform(1.0 - JITTER_RANGE, 1.0 + JITTER_RANGE)
            if attempt >= 2 and label:
                print(f"    ⚠ {label}: {err_type}, retry {attempt+1}/{max_retries} in {delay:.1f}s")
            await asyncio.sleep(delay)
        else:
            if label:
                print(f"    ✗ {label}: {err_type}, all {max_retries} retries exhausted")
    return None


# ============================================================================
# GENERATION FUNCTIONS — with retry + backoff
# ============================================================================
semaphore = asyncio.Semaphore(CONCURRENCY)

async def generate_questions_for_chunk(chunk: str, round_idx: int, voice_mode: str, chunk_idx: int = 0) -> list[str]:
    """Generate questions for a chunk — with retry on failure."""
    prompt = QUESTION_PROMPTS[round_idx % len(QUESTION_PROMPTS)].format(
        n=QUESTIONS_PER_CHUNK
    )
    async with semaphore:
        def make_call():
            return client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": prompt},
                    {"role": "user", "content": chunk},
                ],
                temperature=TEMPERATURE_QUESTIONS,
                max_tokens=1024,
                response_format={"type": "json_object"},
            )
        resp = await _retry_api_call(make_call, label=f"Q-chunk{chunk_idx}")
        if resp is None:
            return []
        try:
            text = resp.choices[0].message.content
            del resp
            text = re.sub(r'^```json\s*', '', text.strip())
            text = re.sub(r'\s*```$', '', text.strip())
            result = json.loads(text)
            return result.get("questions", [])[:QUESTIONS_PER_CHUNK]
        except (json.JSONDecodeError, IndexError, AttributeError) as e:
            _error_counts["json_parse"] += 1
            return []

async def _single_answer_call(system_prompt: str, user_prompt: str, label: str = "") -> str:
    """Make one answer API call with retry."""
    async with semaphore:
        def make_call():
            return client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=TEMPERATURE_ANSWERS,
                max_tokens=1024,
                frequency_penalty=0.5,
                presence_penalty=0.2,
            )
        resp = await _retry_api_call(make_call, label=label)
        if resp is None:
            return ""
        try:
            answer = resp.choices[0].message.content.strip()
            del resp
            return answer
        except (IndexError, AttributeError):
            return ""

async def generate_answer(question: str, chunk: str, voice_mode: str, idx: int = 0) -> str:
    """Generate an answer in the persona's voice for the given voice mode.
    
    Retries once OUTSIDE the semaphore if template detected.
    """
    system_prompt = make_system_prompt(voice_mode)

    # Get voice-mode-specific opener cues and randomly select 3 for variety
    mode_data = VOICE_MODES.get(voice_mode, {})
    opener_cues = mode_data.get("opener_cues", [])

    if opener_cues:
        selected = random.sample(opener_cues, min(3, len(opener_cues)))
        cue_lines = "\n".join(f"  • {c}" for c in selected)
        opener_instruction = (
            f"For THIS specific response, try one of these opening approaches:\n"
            f"{cue_lines}\n"
            f"Pick whichever fits the question best. Do NOT reuse the same opening "
            f"pattern you used in previous answers."
        )
    else:
        opener_instruction = (
            "Start with something only YOU would say in this voice mode — a vivid image from your life, "
            "a characteristic phrase, a direct answer, a rhetorical question in your style, "
            "or a reference to a specific event you experienced."
        )

    user_prompt = (
        f"Use the following passage to inform your answer, but respond naturally "
        f"as yourself — do not quote it directly or reference 'a text':\n"
        f"---\n{chunk}\n---\n\n"
        f"Question: {question}\n\n"
        f"CRITICAL: Your opening sentence must be completely unique and specific to this answer. "
        f"Do NOT begin with any of these: 'The weight of', 'My friend', 'The memory', "
        f"'The memories', 'My child', 'I remember', 'I recall', 'You see', 'Ah', "
        f"'Brother', 'Friend', 'Let me tell you', 'Well'.\n\n"
        f"{opener_instruction}"
    )

    # Attempt 1 — release semaphore fully before any retry
    answer = await _single_answer_call(system_prompt, user_prompt, label=f"A-{idx}")

    # Retry once if template detected (semaphore was released, so no deadlock)
    if answer and is_template_answer(answer):
        answer = await _single_answer_call(system_prompt, user_prompt, label=f"A-{idx}-tmpl")

    return answer

def _partial_path(ckpt_dir: str, label: str, round_idx: int) -> str:
    """Path for a per-round partial file: e.g. liguori_work.r0.partial.jsonl"""
    return f"{ckpt_dir}/{label}.r{round_idx}.partial.jsonl"

def _done_path(ckpt_dir: str, label: str, round_idx: int) -> str:
    """Path for round completion marker: e.g. liguori_work.r0.done"""
    return f"{ckpt_dir}/{label}.r{round_idx}.done"

def _count_lines(path: str) -> int:
    if not os.path.exists(path):
        return 0
    with open(path) as f:
        return sum(1 for _ in f)

def _mark_round_done(ckpt_dir: str, label: str, round_idx: int):
    """Create a marker file indicating a round has been fully processed."""
    Path(_done_path(ckpt_dir, label, round_idx)).touch()

def _is_round_done(ckpt_dir: str, label: str, round_idx: int) -> bool:
    """Check if a round was fully processed.
    A round is done if it has a .done marker OR a non-empty partial file.
    (Partial files are only written after all API calls for that round finish.)
    """
    if os.path.exists(_done_path(ckpt_dir, label, round_idx)):
        return True
    pf = _partial_path(ckpt_dir, label, round_idx)
    return os.path.exists(pf) and _count_lines(pf) > 0

def _merge_partials(ckpt_dir: str, label: str, outfile: str, num_rounds: int):
    """Merge per-round partial files into the final source .jsonl, then delete partials and markers."""
    with open(outfile, "w") as out:
        for r in range(num_rounds):
            pf = _partial_path(ckpt_dir, label, r)
            if os.path.exists(pf):
                with open(pf) as inp:
                    for line in inp:
                        out.write(line)
                os.remove(pf)
            done = _done_path(ckpt_dir, label, r)
            if os.path.exists(done):
                os.remove(done)

# ── INTER-SOURCE COOLDOWN ──
SOURCE_COOLDOWN = 15        # seconds between sources to let rate limits recover

# ── Process ONE SOURCE FILE AT A TIME: read → chunk → Q → A → write per round → merge ──
qa_dir = f"{OUTPUT_DIR}/per_source"
os.makedirs(qa_dir, exist_ok=True)
ckpt_dir = f"{qa_dir}/_checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)
grand_total = 0
template_reject_total = 0

for source_idx, entry in enumerate(source_registry):
    filepath = entry["filepath"]
    voice_mode = entry["voice_mode"]
    label = entry["label"]
    outfile = f"{qa_dir}/{label}.jsonl"

    # ── Resume check: final file already exists with content? ACCEPT IT. ──
    if os.path.exists(outfile):
        existing = _count_lines(outfile)
        if existing > 0:
            print(f"  {label:30s} [{voice_mode:14s}] SKIP ({existing} QA pairs — already generated)")
            grand_total += existing
            continue
        else:
            print(f"  {label:30s} [{voice_mode:14s}] EMPTY final file — removing, will check partials")
            os.remove(outfile)

    # ── Figure out which rounds are already done (partial files / done markers) ──
    rounds_done = {}
    for r in range(NUM_ROUNDS):
        if _is_round_done(ckpt_dir, label, r):
            pf = _partial_path(ckpt_dir, label, r)
            count = _count_lines(pf) if os.path.exists(pf) else 0
            rounds_done[r] = count

    if len(rounds_done) == NUM_ROUNDS:
        total_partial = sum(rounds_done.values())
        print(f"  {label:30s} [{voice_mode:14s}] All rounds done in partials ({total_partial} QA) — merging")
        _merge_partials(ckpt_dir, label, outfile, NUM_ROUNDS)
        grand_total += total_partial
        continue

    # ── Inter-source cooldown (skip for the first source) ──
    if source_idx > 0:
        print(f"\n  ⏳ Cooling down {SOURCE_COOLDOWN}s before next source (rate limit recovery)...")
        await asyncio.sleep(SOURCE_COOLDOWN)

    # Read and chunk THIS source file only
    with open(filepath) as f:
        text = f.read()
    chunks = chunk_text(text)
    del text

    # Apply test limit if set
    if TEST_CHUNKS_PER_ROUND:
        chunks = chunks[:TEST_CHUNKS_PER_ROUND]

    rounds_to_run = [r for r in range(NUM_ROUNDS) if r not in rounds_done]
    skipped_total = sum(rounds_done.values())

    # Calculate expected values for display
    expected_qa = len(chunks) * QUESTIONS_PER_CHUNK * NUM_ROUNDS
    expected_per_round = len(chunks) * QUESTIONS_PER_CHUNK

    # Reset error counters for this source
    _error_counts.clear()

    print(f"\n{'='*70}")
    test_tag = f" [TEST MODE: {len(chunks)} chunks]" if TEST_CHUNKS_PER_ROUND else ""
    print(f"  {label.upper()} (catechetical) — {len(chunks)} chunks × {QUESTIONS_PER_CHUNK} Q/chunk{test_tag}")
    print(f"  Rounds to run: {rounds_to_run} (skipping {list(rounds_done.keys())} with {skipped_total} QA on disk)")
    print(f"  Expected total: ~{expected_qa} QA pairs")
    print(f"{'='*70}")

    for round_idx in range(NUM_ROUNDS):
        round_name = ['Factual', 'Application', 'Reflective'][round_idx % 3]
        pf = _partial_path(ckpt_dir, label, round_idx)

        if round_idx in rounds_done:
            print(f"  {label} R{round_idx+1} ({round_name}) — SKIP ({rounds_done[round_idx]} QA on disk)")
            continue

        # Generate questions — voice-mode-aware, with chunk index for error labels
        q_tasks = [generate_questions_for_chunk(c, round_idx, voice_mode, chunk_idx=i) for i, c in enumerate(chunks)]
        q_results = await atqdm.gather(*q_tasks, desc=f"  {label} R{round_idx+1} ({round_name}) Q")

        qa_batch = []
        total_questions = 0
        empty_chunks = 0
        for chunk, questions in zip(chunks, q_results):
            if not questions:
                empty_chunks += 1
            for q in questions:
                q = q.strip()
                if len(q) > 15:
                    qa_batch.append({"chunk": chunk, "question": q})
                    total_questions += 1

        # ── YIELD DIAGNOSTIC ──
        q_yield_pct = (total_questions / expected_per_round * 100) if expected_per_round else 0
        yield_status = "✓" if q_yield_pct > 80 else "⚠" if q_yield_pct > 40 else "✗"
        print(f"  {yield_status} Q yield: {total_questions}/{expected_per_round} ({q_yield_pct:.0f}%) — "
              f"{empty_chunks}/{len(chunks)} chunks returned 0 questions")

        if total_questions == 0:
            print(f"    ✗ No questions generated — skipping answer phase for this round")
            print(f"    Error breakdown: {dict(_error_counts)}")
            _mark_round_done(ckpt_dir, label, round_idx)
            with open(pf, "w") as f:
                pass
            continue

        del q_tasks, q_results
        gc.collect()

        # ── Small cooldown between Q and A phases to ease rate pressure ──
        if q_yield_pct < 80:
            cooldown = 10
            print(f"    ⏳ Low Q yield — cooling down {cooldown}s before answer phase...")
            await asyncio.sleep(cooldown)

        # Generate answers with template rejection
        a_tasks = [generate_answer(qa["question"], qa["chunk"], voice_mode, idx=i) for i, qa in enumerate(qa_batch)]
        a_results = await atqdm.gather(*a_tasks, desc=f"  {label} R{round_idx+1} ({round_name}) A")
        del a_tasks

        # Write results — filter out template answers AND short answers
        round_count = 0
        round_template_rejects = 0
        round_empty = 0
        with open(pf, "w") as f:
            for qa, answer in zip(qa_batch, a_results):
                if len(answer) < 20:
                    round_empty += 1
                    continue
                if is_template_answer(answer):
                    round_template_rejects += 1
                    continue
                item = {
                    "persona": "church_tradition",
                    "voice_mode": voice_mode,
                    "source_label": label,
                    "question": qa["question"],
                    "answer": answer,
                    "chunk_key": qa["chunk"][:100],
                }
                f.write(json.dumps(item) + "\n")
                round_count += 1

        # Mark round as fully processed
        _mark_round_done(ckpt_dir, label, round_idx)

        template_reject_total += round_template_rejects
        a_yield_pct = (round_count / total_questions * 100) if total_questions else 0
        print(f"  ✓ {label} R{round_idx+1}: {round_count}/{total_questions} QA saved "
              f"(A yield: {a_yield_pct:.0f}%, {round_empty} empty, {round_template_rejects} template) → {pf}")
        del qa_batch, a_results
        gc.collect()

        # Inter-round cooldown
        if round_idx < NUM_ROUNDS - 1:
            await asyncio.sleep(5)

    # All rounds done — merge partials into final file
    _merge_partials(ckpt_dir, label, outfile, NUM_ROUNDS)
    count = _count_lines(outfile)
    grand_total += count
    print(f"  ✓ {label}: {count}/{expected_qa} QA pairs merged → {outfile}")
    del chunks
    gc.collect()
    print(f"  🧹 Memory cleared for {label}")

    # ── Source-level error summary ──
    if any(_error_counts.values()):
        print(f"  📊 Errors this source: {dict(_error_counts)}")

print(f"\n{'='*70}")
print(f"DONE: {grand_total:,} total QA pairs across {len(source_registry)} source files")
print(f"Template answers rejected: {template_reject_total:,}")
print(f"Per-source files in: {qa_dir}/")

## 5b. Quality Gate

**Run BEFORE assembly.**

In [ ]:
from collections import Counter

qa_dir = f"{OUTPUT_DIR}/per_source"
qa_files = sorted(glob.glob(f"{qa_dir}/*.jsonl"))

print("VOICE QUALITY GATE\n")
print(f"{'='*70}")

all_openers = []
global_opener_counts = Counter()
total_all = 0
template_all = 0

for qa_file in qa_files:
    with open(qa_file) as f:
        for line in f:
            item = json.loads(line)
            answer = item["answer"].strip()
            total_all += 1
            if is_template_answer(answer):
                template_all += 1
            opener = ' '.join(answer.split()[:4])
            all_openers.append(opener)
            global_opener_counts[opener] += 1

global_contam = (template_all / total_all * 100) if total_all else 0
print(f"\n  Total QA pairs: {total_all}")
print(f"  Template contamination: {template_all} ({global_contam:.1f}%)")

print(f"\n{'='*70}")
print("Top 10 most repeated opening 4-grams:")
for phrase, count in global_opener_counts.most_common(10):
    print(f'  {count:>4}x: "{phrase}"')

unique_openers = sum(1 for o in all_openers if global_opener_counts[o] == 1)
pct = unique_openers / len(all_openers) * 100 if all_openers else 0
print(f"\nOpener uniqueness: {pct:.0f}% ({unique_openers}/{len(all_openers)})")

print(f"\n{'='*70}")
if global_contam > 30:
    print("✗ QUALITY GATE FAILED — template contamination too high ({:.1f}%)".format(global_contam))
    QUALITY_GATE_PASSED = False
elif global_contam > 15:
    print("⚠ QUALITY GATE WARNING — elevated ({:.1f}%)".format(global_contam))
    QUALITY_GATE_PASSED = True
else:
    print("✓ QUALITY GATE PASSED — {:.1f}% (target: <15%)".format(global_contam))
    QUALITY_GATE_PASSED = True

del all_openers, global_opener_counts

## 6. Assemble Conversations & Save

All conversations use the **catechetical** system prompt.

In [ ]:
if not QUALITY_GATE_PASSED:
    raise RuntimeError("Quality gate FAILED. Delete bad per_source/*.jsonl files and re-run.")

def quality_check(conv):
    for msg in conv["conversations"]:
        if msg["from"] == "gpt":
            v = msg["value"]
            if len(v) < 30: return False
            lower = v.lower()
            if any(p in lower for p in ["as an ai", "as a language model", "i cannot fulfill", "i'm sorry, but"]): return False
            if is_template_answer(v): return False
    return True

total_convs = 0
template_filtered = 0
qa_dir = f"{OUTPUT_DIR}/per_source"
qa_files = sorted(glob.glob(f"{qa_dir}/*.jsonl"))
print(f"Reading {len(qa_files)} per-source files\n")

with open(OUTPUT_FILE, "w") as out_f:
    for qa_file in qa_files:
        label = Path(qa_file).stem
        items = []
        with open(qa_file) as f:
            for line in f:
                items.append(json.loads(line))
        if not items:
            print(f"  {label:30s}    0 QA →    0 conversations (empty)")
            continue
        groups = defaultdict(list)
        for item in items:
            groups[item["chunk_key"]].append(item)
        source_count = 0
        for _, group_items in groups.items():
            random.shuffle(group_items)
            for i in range(0, len(group_items), TURNS_PER_CONVERSATION):
                batch = group_items[i:i + TURNS_PER_CONVERSATION]
                if len(batch) < 2: continue
                conv = {"conversations": [
                    {"from": "system", "value": make_system_prompt("catechetical")}
                ]}
                for qa in batch:
                    conv["conversations"].append({"from": "human", "value": qa["question"]})
                    conv["conversations"].append({"from": "gpt", "value": qa["answer"]})
                if quality_check(conv):
                    out_f.write(json.dumps(conv) + "\n")
                    source_count += 1
                else:
                    template_filtered += 1
        total_convs += source_count
        print(f"  {label:30s} {len(items):>5} QA → {source_count:>4} conversations")
        del items, groups
        gc.collect()

print(f"\n✓ Saved {total_convs:,} conversations to:")
print(f"  {OUTPUT_FILE}")
print(f"  ({os.path.getsize(OUTPUT_FILE) / 1024 / 1024:.1f} MB)")
if template_filtered:
    print(f"  (filtered {template_filtered} conversations)")

import subprocess
subprocess.run(["shuf", OUTPUT_FILE, "-o", OUTPUT_FILE])
print(f"  ✓ Shuffled output file")

## 7. Verify

In [ ]:
total_turns = 0
total_convs_v = 0
sample_convs = []

with open(OUTPUT_FILE) as f:
    for line_num, line in enumerate(f):
        conv = json.loads(line)
        total_convs_v += 1
        total_turns += len(conv["conversations"]) - 1
        if len(sample_convs) < 3:
            sample_convs.append(conv)
        else:
            j = random.randint(0, line_num)
            if j < 3: sample_convs[j] = conv
        del conv

print(f"{'='*50}")
print(f"TOTAL: {total_convs_v:,} conversations, {total_turns:,} turns ({total_turns//2:,} QA pairs)")
print(f"Persona: church_tradition (single collective persona, catechetical mode)")

print(f"\n{'='*50}")
print("SAMPLE CONVERSATIONS:\n")
for conv in sample_convs:
    print(f"──────────────────────────────────────────────────")
    for msg in conv["conversations"]:
        role = msg["from"].upper()
        text = msg["value"][:250] + ("..." if len(msg["value"]) > 250 else "")
        print(f"[{role}] {text}\n")

del sample_convs
gc.collect()

print(f"\n✓ Data ready for training.")
print(f"  {OUTPUT_FILE}")